# Brian LightGBM v4

Professional reassessment after the hidden Secret Set 2 review.

V4 goals:
- Keep the upside capture that made v3 strong on biotech-like hidden sets.
- Reduce benchmark beta, crash-day exposure, and drawdown behavior.
- Select portfolio controls on validation only, then run test once with locked settings.
- Prove whether the ranker adds value beyond momentum, volatility, and simple risk filters.

Submission rule for this week: this notebook is the only file intended for PR.


In [1]:
import os
import sys
from pathlib import Path
import json
import math
from copy import deepcopy

import numpy as np
import pandas as pd
import lightgbm as lgb


def is_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def find_repo_root() -> Path:
    candidates = [Path.cwd()] + list(Path.cwd().parents)
    for path in candidates:
        if is_repo_root(path):
            return path
    raise RuntimeError("Could not find repo root. Run this notebook from inside the repository.")


repo_root = find_repo_root()
os.chdir(repo_root)
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from portfolio_toolkit import (
    PortfolioWeights,
    backtest_weights,
    baseline_weights,
    build_features,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    split_dates,
    start_run,
    validate_feature_frame,
    validate_prediction_frame,
    validate_weights_frame,
    write_backtest_artifacts,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

print("repo_root =", repo_root)
print("Imports successful.")


repo_root = C:\Users\brixn\Documents\Portfolio-Optimization-Lib
Imports successful.


In [50]:
# Configuration
DATASET_NAME = "shared_set_1"
MODEL_VERSION = "v4"
MODEL_NAME = "Brian_lgbm_v4_regime_beta_controlled"
STRATEGY_NAME = "brian_lgbm_v4_regime_beta_controlled"

HORIZON = 10
EMBARGO_DAYS = HORIZON
N_LABEL_BINS = 10
REBALANCE_FREQUENCY = "weekly_first_trading_day"
RANDOM_SEED = 42
ENSEMBLE_SEEDS = [17, 42, 101]

# Universe-relative construction. This is more robust on random hidden subsets than fixed top-N.
BASE_TOP_FRACTION = 0.45
STRESS_TOP_FRACTION = 0.55
MIN_HOLDINGS = 50
MAX_HOLDINGS = 180
BASE_MAX_WEIGHT = 0.025
STRESS_MAX_WEIGHT = 0.020
TURNOVER_BLEND = 0.65
BASE_RISK_POWER = 0.35
STRESS_RISK_POWER = 0.65
BETA_TARGET = 1.05
BETA_PENALTY = 0.08
ENTRY_QUANTILE = 0.50

# Keep off for PR cleanliness. Turn on only when intentionally logging.
RUN_MLFLOW = False
SAVE_LOCAL_ARTIFACTS = False
RUN_ROBUSTNESS_DATASETS = False
ROBUSTNESS_DATASETS = ["shared_set_2", "shared_set_3"]

output_dir = repo_root / "runs" / "Brian_lgbm_v4"
notebook_path = repo_root / "MODELS" / "Brian" / "brian_lgbm_v4.ipynb"
model_dir = repo_root / "MODELS" / "Brian" / "v4_artifacts"

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
SPLITS = split_dates(DATASET_NAME, repo_root=repo_root)
TRAIN_START, TRAIN_END = SPLITS["train"]
VAL_START, VAL_END = SPLITS["val"]

# Local final-test proxy for this week's likely hidden-data window.
# Adam said the hidden data may end around 2022, so do not tune on 2023-2025.
TEST_START = pd.Timestamp("2022-01-03")
TEST_END = pd.Timestamp("2022-12-31")


print("Dataset:", DATASET_NAME, spec.name)
print("Horizon:", HORIZON)
print("Rebalance:", REBALANCE_FREQUENCY)
print("Train:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Val:  ", VAL_START.date(), "to", VAL_END.date())
print("Test: ", TEST_START.date(), "to", TEST_END.date())


Dataset: shared_set_1 sp500_full_universe
Horizon: 10
Rebalance: weekly_first_trading_day
Train: 2014-01-02 to 2019-12-31
Val:   2020-01-02 to 2021-12-31
Test:  2022-01-03 to 2022-12-31


In [3]:
# Data and decision calendar
prices = load_prices(DATASET_NAME, repo_root=repo_root)
prices["date"] = pd.to_datetime(prices["date"], utc=True).dt.tz_localize(None)
prices["ticker"] = prices["ticker"].astype(str).str.upper()
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)
TRADING_DATES = pd.DatetimeIndex(sorted(prices["date"].unique()))


def trading_day_offset(date_value, offset: int, trading_dates: pd.DatetimeIndex = TRADING_DATES) -> pd.Timestamp:
    date_value = pd.Timestamp(date_value)
    position = int(trading_dates.searchsorted(date_value, side="left"))
    if position >= len(trading_dates):
        position = len(trading_dates) - 1
    target = min(max(position + int(offset), 0), len(trading_dates) - 1)
    return pd.Timestamp(trading_dates[target])


def build_decision_calendar(prices_frame: pd.DataFrame) -> pd.DataFrame:
    dates = pd.DatetimeIndex(sorted(pd.to_datetime(prices_frame["date"], utc=True).dt.tz_localize(None).unique()))
    first_by_week = (
        pd.DataFrame({"execution_date": dates})
        .groupby(pd.Series(dates).dt.to_period("W"), sort=True)["execution_date"]
        .first()
    )
    records = []
    for execution_date in pd.DatetimeIndex(first_by_week.to_numpy()):
        position = dates.searchsorted(execution_date, side="left")
        if position <= 0:
            continue
        records.append({
            "signal_date": pd.Timestamp(dates[position - 1]),
            "date": pd.Timestamp(execution_date),
        })
    return pd.DataFrame(records).drop_duplicates("date").sort_values("date").reset_index(drop=True)


def date_window(frame: pd.DataFrame, start, end, column: str = "date") -> pd.DataFrame:
    return frame.loc[(frame[column] >= pd.Timestamp(start)) & (frame[column] <= pd.Timestamp(end))].copy()


decision_calendar = build_decision_calendar(prices)
decision_calendar = date_window(decision_calendar, TRAIN_START, TEST_END, column="date")

print("Prices loaded:", prices.shape)
print("Price range:", prices["date"].min().date(), "to", prices["date"].max().date())
print("Decision dates:", len(decision_calendar), decision_calendar["date"].min().date(), "to", decision_calendar["date"].max().date())
print(decision_calendar.head().to_string(index=False))


Prices loaded: (1463605, 8)
Price range: 2014-01-02 to 2025-12-31
Decision dates: 469 2014-01-06 to 2022-12-27
signal_date       date
 2014-01-03 2014-01-06
 2014-01-10 2014-01-13
 2014-01-17 2014-01-21
 2014-01-24 2014-01-27
 2014-01-31 2014-02-03


In [4]:
# Feature factory with market context and cross-sectional transforms
TOOLKIT_FEATURES = [
    "return_1d", "return_5d", "return_10d", "return_20d", "return_60d",
    "vol_20d", "vol_60d", "downside_vol_20d", "upside_vol_20d",
    "beta_20d_spy", "beta_60d_spy",
    "momentum_5d", "momentum_20d", "momentum_60d", "momentum_120d",
    "price_to_sma_20d", "price_to_sma_50d", "macd_hist",
    "rsi_14", "bollinger_z_20d",
    "excess_return_5d_vs_spy", "excess_return_20d_vs_spy", "excess_return_60d_vs_spy",
    "relative_momentum_20d_vs_spy",
    "volume_zscore_20d", "volume_zscore_60d", "dollar_volume_ratio_20d",
    "distance_to_20d_high", "distance_to_60d_high",
]

CUSTOM_FEATURES = [
    "mom_vol_ratio", "downside_ratio", "mom_divergence", "mom_60_20_divergence",
    "mom_beta_adjusted", "trend_strength", "residual_momentum_20d", "residual_momentum_60d",
    "beta_instability", "vol_compression", "short_reversal_5d", "drawdown_reversal_setup",
]

MARKET_FEATURES = [
    "spy_return_5d", "spy_return_10d", "spy_return_20d", "spy_return_60d",
    "spy_vol_20d", "spy_vol_60d", "spy_drawdown_60d", "spy_drawdown_120d",
    "spy_price_to_sma_20d", "spy_price_to_sma_50d",
    "breadth_above_sma_20d", "breadth_above_sma_50d",
    "return_dispersion_20d", "avg_corr_to_spy_60d",
]

CS_BASE_FEATURES = [
    "momentum_20d", "momentum_60d", "momentum_120d", "vol_20d", "vol_60d",
    "beta_60d_spy", "excess_return_20d_vs_spy", "excess_return_60d_vs_spy",
    "distance_to_60d_high", "volume_zscore_60d", "residual_momentum_20d", "vol_compression",
]
CS_FEATURES = [f"cs_rank_{name}" for name in CS_BASE_FEATURES] + [f"cs_z_{name}" for name in CS_BASE_FEATURES]
FEATURE_COLS = TOOLKIT_FEATURES + CUSTOM_FEATURES + MARKET_FEATURES + CS_FEATURES


def build_market_context(prices_ref: pd.DataFrame) -> pd.DataFrame:
    panel = prices_ref.sort_values(["ticker", "date"]).copy()
    panel["return_1d_local"] = panel.groupby("ticker", sort=False)["adj_close"].pct_change()
    wide_returns = panel.pivot(index="date", columns="ticker", values="return_1d_local").sort_index()

    spy = panel.loc[panel["ticker"] == "SPY", ["date", "adj_close"]].drop_duplicates("date").set_index("date")["adj_close"].sort_index()
    market = pd.DataFrame(index=spy.index)
    for horizon in [5, 10, 20, 60]:
        market[f"spy_return_{horizon}d"] = spy.pct_change(horizon)
    spy_daily = spy.pct_change()
    market["spy_vol_20d"] = spy_daily.rolling(20, min_periods=20).std(ddof=0)
    market["spy_vol_60d"] = spy_daily.rolling(60, min_periods=60).std(ddof=0)
    market["spy_drawdown_60d"] = spy / spy.rolling(60, min_periods=60).max() - 1.0
    market["spy_drawdown_120d"] = spy / spy.rolling(120, min_periods=120).max() - 1.0
    market["spy_price_to_sma_20d"] = spy / spy.rolling(20, min_periods=20).mean() - 1.0
    market["spy_price_to_sma_50d"] = spy / spy.rolling(50, min_periods=50).mean() - 1.0

    tradable = panel.loc[panel["ticker"] != "SPY", ["date", "ticker", "adj_close", "return_1d_local"]].copy()
    tradable["sma_20"] = tradable.groupby("ticker", sort=False)["adj_close"].transform(lambda s: s.rolling(20, min_periods=20).mean())
    tradable["sma_50"] = tradable.groupby("ticker", sort=False)["adj_close"].transform(lambda s: s.rolling(50, min_periods=50).mean())
    tradable["above_sma_20"] = tradable["adj_close"] > tradable["sma_20"]
    tradable["above_sma_50"] = tradable["adj_close"] > tradable["sma_50"]
    market["breadth_above_sma_20d"] = tradable.groupby("date")["above_sma_20"].mean()
    market["breadth_above_sma_50d"] = tradable.groupby("date")["above_sma_50"].mean()

    tradable_cols = [col for col in wide_returns.columns if col != "SPY"]
    market["return_dispersion_20d"] = wide_returns[tradable_cols].std(axis=1, skipna=True).rolling(20, min_periods=20).mean()
    corr_to_spy = wide_returns[tradable_cols].rolling(60, min_periods=40).corr(wide_returns.get("SPY"))
    market["avg_corr_to_spy_60d"] = corr_to_spy.mean(axis=1, skipna=True)

    return market.reset_index().rename(columns={"index": "date"})


def add_custom_features(features: pd.DataFrame, prices_ref: pd.DataFrame) -> pd.DataFrame:
    features = features.copy()
    features["mom_vol_ratio"] = features["momentum_20d"] / features["vol_20d"].replace(0.0, np.nan)
    features["downside_ratio"] = features["downside_vol_20d"] / features["vol_20d"].replace(0.0, np.nan)
    features["mom_divergence"] = features["momentum_5d"] - features["momentum_20d"]
    features["mom_60_20_divergence"] = features["momentum_60d"] - features["momentum_20d"]
    features["mom_beta_adjusted"] = features["momentum_20d"] / features["beta_20d_spy"].replace(0.0, np.nan)
    features["trend_strength"] = features["price_to_sma_20d"] * features["momentum_20d"]
    features["residual_momentum_20d"] = features["momentum_20d"] - features["beta_20d_spy"] * features["spy_return_20d"]
    features["residual_momentum_60d"] = features["momentum_60d"] - features["beta_60d_spy"] * features["spy_return_60d"]
    features["beta_instability"] = features["beta_20d_spy"] - features["beta_60d_spy"]
    features["vol_compression"] = features["vol_20d"] / features["vol_60d"].replace(0.0, np.nan)
    features["short_reversal_5d"] = -features["return_5d"]
    features["drawdown_reversal_setup"] = features["short_reversal_5d"] * features["distance_to_60d_high"].abs()
    return features


def add_cross_sectional_features(features: pd.DataFrame) -> pd.DataFrame:
    features = features.copy()
    for name in CS_BASE_FEATURES:
        features[f"cs_rank_{name}"] = features.groupby("date")[name].rank(pct=True, method="average")
        mean = features.groupby("date")[name].transform("mean")
        std = features.groupby("date")[name].transform(lambda values: values.std(ddof=0))
        features[f"cs_z_{name}"] = (features[name] - mean) / std.replace(0.0, np.nan)
    return features


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    base = build_features(prices, feature_names=TOOLKIT_FEATURES)
    market = build_market_context(prices)
    features = base.merge(market, on="date", how="left")
    features = add_custom_features(features, prices)
    features = add_cross_sectional_features(features)
    features[FEATURE_COLS] = features[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    return validate_feature_frame(features)


features = build_model_features(prices)
feature_audit = features[FEATURE_COLS].isna().mean().sort_values(ascending=False).rename("missing_rate").reset_index().rename(columns={"index": "feature"})
leaky_features = [name for name in FEATURE_COLS if name.startswith("forward_")]
if leaky_features:
    raise ValueError(f"Forward-looking feature columns found: {leaky_features}")

print("Features built:", features.shape)
print("Feature count:", len(FEATURE_COLS))
print("Top missing rates:")
print(feature_audit.head(15).to_string(index=False))


Features built: (1463605, 81)
Feature count: 79
Top missing rates:
                 feature  missing_rate
           momentum_120d      0.041272
   cs_rank_momentum_120d      0.041272
      cs_z_momentum_120d      0.041272
       spy_drawdown_120d      0.037271
         vol_compression      0.020657
    cs_z_vol_compression      0.020657
 cs_rank_vol_compression      0.020657
excess_return_60d_vs_spy      0.020652
    mom_60_20_divergence      0.020652
                 vol_60d      0.020652
              return_60d      0.020652
            beta_60d_spy      0.020652
            momentum_60d      0.020652
    cs_rank_momentum_60d      0.020652
    cs_rank_beta_60d_spy      0.020652


In [5]:
# Label factory. V4 labels start at execution date, not signal date.
TARGET_COL = f"forward_alpha_{HORIZON}d_vs_spy"
LABEL_COL = "alpha_rank_label"


def add_target_labels(targets: pd.DataFrame) -> pd.DataFrame:
    labeled = targets.copy()
    labeled = labeled[labeled["ticker"].str.upper() != "SPY"].copy()
    labeled["alpha_rank_pct"] = labeled.groupby("date")[TARGET_COL].rank(pct=True, method="average")
    labeled["alpha_zscore"] = labeled.groupby("date")[TARGET_COL].transform(
        lambda values: (values - values.mean()) / (values.std(ddof=0) + 1e-12)
    )
    labeled["top_quintile"] = (labeled["alpha_rank_pct"] >= 0.80).astype(float)
    labeled["top_decile"] = (labeled["alpha_rank_pct"] >= 0.90).astype(float)
    raw_label = np.floor(labeled["alpha_rank_pct"] * N_LABEL_BINS)
    labeled[LABEL_COL] = raw_label.clip(lower=0, upper=N_LABEL_BINS - 1)
    labeled.loc[labeled["alpha_rank_pct"].isna(), LABEL_COL] = np.nan
    return labeled


targets = make_forward_alpha_target(prices, horizon=HORIZON)
targets = add_target_labels(targets)

feature_decisions = (
    features
    .rename(columns={"date": "signal_date"})
    .merge(decision_calendar, on="signal_date", how="inner")
)
feature_decisions = feature_decisions[feature_decisions["ticker"].str.upper() != "SPY"].copy()
feature_decisions = feature_decisions.sort_values(["date", "ticker"]).reset_index(drop=True)

model_frame = feature_decisions.merge(
    targets[["date", "ticker", TARGET_COL, "alpha_rank_pct", "alpha_zscore", "top_quintile", "top_decile", LABEL_COL]],
    on=["date", "ticker"],
    how="left",
)
model_frame = model_frame.dropna(subset=[TARGET_COL, LABEL_COL]).copy()
model_frame[LABEL_COL] = model_frame[LABEL_COL].astype(int)
model_frame = model_frame.sort_values(["date", "ticker"]).reset_index(drop=True)

print("Decision feature rows:", feature_decisions.shape)
print("Labeled model rows:", model_frame.shape)
print("Unique execution dates:", model_frame["date"].nunique())
print("Label distribution:")
print(model_frame[LABEL_COL].value_counts().sort_index().to_string())
print("Timing audit:")
print(model_frame[["signal_date", "date", "ticker", TARGET_COL, LABEL_COL]].head().to_string(index=False))


Decision feature rows: (224439, 82)
Labeled model rows: (224439, 88)
Unique execution dates: 469
Label distribution:
alpha_rank_label
0    22167
1    22476
2    22433
3    22449
4    22314
5    22555
6    22491
7    22386
8    22520
9    22648
Timing audit:
signal_date       date ticker  forward_alpha_10d_vs_spy  alpha_rank_label
 2014-01-03 2014-01-06      A                  0.064348                 8
 2014-01-03 2014-01-06   AAPL                 -0.000531                 3
 2014-01-03 2014-01-06   ABBV                 -0.009860                 3
 2014-01-03 2014-01-06    ABT                 -0.005160                 3
 2014-01-03 2014-01-06   ACGL                 -0.029715                 1


In [6]:
# Purged walk-forward folds

def purged_train_validation(frame: pd.DataFrame, val_start, val_end, embargo_days: int = EMBARGO_DAYS) -> tuple[pd.DataFrame, pd.DataFrame, pd.Timestamp]:
    val_start = pd.Timestamp(val_start)
    val_end = pd.Timestamp(val_end)
    train_cutoff = trading_day_offset(val_start, -int(embargo_days), TRADING_DATES)
    train = frame.loc[frame["date"] <= train_cutoff].copy()
    val = frame.loc[(frame["date"] >= val_start) & (frame["date"] <= val_end)].copy()
    return train, val, train_cutoff


def build_walk_forward_folds(frame: pd.DataFrame) -> list[dict]:
    folds = []
    for year in [2018, 2019, 2020, 2021]:
        val_start = max(pd.Timestamp(f"{year}-01-01"), TRAIN_START)
        val_end = min(pd.Timestamp(f"{year}-12-31"), VAL_END)
        if val_start > val_end:
            continue
        train, val, cutoff = purged_train_validation(frame, val_start, val_end)
        if train.empty or val.empty:
            continue
        folds.append({
            "name": f"wf_{year}",
            "val_start": val_start,
            "val_end": val_end,
            "train_cutoff": cutoff,
            "train": train,
            "val": val,
        })
    return folds


walk_forward_folds = build_walk_forward_folds(model_frame)
for fold in walk_forward_folds:
    print(
        fold["name"],
        "train_rows=", len(fold["train"]),
        "val_rows=", len(fold["val"]),
        "train_dates=", fold["train"]["date"].nunique(),
        "val_dates=", fold["val"]["date"].nunique(),
        "purge_cutoff=", fold["train_cutoff"].date(),
    )

final_train_cutoff = trading_day_offset(TEST_START, -EMBARGO_DAYS, TRADING_DATES)
final_train = model_frame.loc[model_frame["date"] <= final_train_cutoff].copy()
test_features = feature_decisions.loc[(feature_decisions["date"] >= TEST_START) & (feature_decisions["date"] <= TEST_END)].copy()
test_labeled = model_frame.loc[(model_frame["date"] >= TEST_START) & (model_frame["date"] <= TEST_END)].copy()

print("Final train rows:", final_train.shape, "through execution", final_train_cutoff.date())
print("Test scoring rows:", test_features.shape)
print("Test labeled rows:", test_labeled.shape)


wf_2018 train_rows= 96200 val_rows= 25306 train_dates= 206 val_dates= 53 purge_cutoff= 2017-12-15
wf_2019 train_rows= 121500 val_rows= 25143 train_dates= 259 val_dates= 52 purge_cutoff= 2018-12-17
wf_2020 train_rows= 146629 val_rows= 25373 train_dates= 311 val_dates= 52 purge_cutoff= 2019-12-17
wf_2021 train_rows= 171992 val_rows= 25674 train_dates= 363 val_dates= 52 purge_cutoff= 2020-12-17
Final train rows: (197658, 88) through execution 2021-12-17
Test scoring rows: (25791, 82)
Test labeled rows: (25791, 88)


In [7]:
# Regime labels and diagnostics helpers
REGIME_FEATURE_COLUMNS = [
    "spy_vol_20d", "spy_drawdown_60d", "breadth_above_sma_50d", "spy_return_20d",
]


def build_regime_thresholds(frame: pd.DataFrame) -> dict[str, float]:
    by_date = frame.drop_duplicates("date")
    return {
        "high_vol": float(by_date["spy_vol_20d"].quantile(0.75)),
        "deep_drawdown": float(by_date["spy_drawdown_60d"].quantile(0.25)),
        "weak_breadth": float(by_date["breadth_above_sma_50d"].quantile(0.25)),
        "strong_rebound": float(by_date["spy_return_20d"].quantile(0.75)),
    }


def add_regime_columns(frame: pd.DataFrame, thresholds: dict[str, float]) -> pd.DataFrame:
    out = frame.copy()
    regime = pd.Series("calm", index=out.index, dtype="object")
    high_vol = out["spy_vol_20d"] >= thresholds["high_vol"]
    drawdown = (out["spy_drawdown_60d"] <= thresholds["deep_drawdown"]) | (out["breadth_above_sma_50d"] <= thresholds["weak_breadth"])
    rebound = (out["spy_return_20d"] >= thresholds["strong_rebound"]) & (out["spy_drawdown_60d"] > thresholds["deep_drawdown"])
    regime.loc[high_vol] = "high_vol"
    regime.loc[drawdown] = "drawdown"
    regime.loc[rebound] = "rebound"
    out["regime"] = regime
    out["stress_flag"] = out["regime"].isin(["high_vol", "drawdown"]).astype(float)
    return out


def score_diagnostics(frame: pd.DataFrame, score_col: str = "model_score", target_col: str = TARGET_COL, group_col: str | None = None) -> pd.DataFrame:
    if target_col not in frame.columns:
        return pd.DataFrame()
    work_cols = ["date", score_col, target_col]
    if group_col is not None and group_col in frame.columns:
        work_cols.append(group_col)
    work = frame[work_cols].replace([np.inf, -np.inf], np.nan).dropna()
    rows = []
    group_iter = [("all", work)] if group_col is None else work.groupby(group_col, sort=True)
    for group_name, group_frame in group_iter:
        date_rows = []
        for date_value, date_frame in group_frame.groupby("date", sort=True):
            if date_frame[score_col].nunique() < 2 or date_frame[target_col].nunique() < 2:
                continue
            rank = date_frame[score_col].rank(pct=True, method="average")
            top = date_frame.loc[rank >= 0.80, target_col]
            bottom = date_frame.loc[rank <= 0.20, target_col]
            date_rows.append({
                "date": date_value,
                "ic": date_frame[score_col].corr(date_frame[target_col], method="pearson"),
                "rank_ic": date_frame[score_col].corr(date_frame[target_col], method="spearman"),
                "top_bottom_spread": top.mean() - bottom.mean() if len(top) and len(bottom) else np.nan,
            })
        if not date_rows:
            continue
        metrics = pd.DataFrame(date_rows)
        rows.append({
            "group": group_name,
            "dates": int(metrics["date"].nunique()),
            "mean_ic": float(metrics["ic"].mean()),
            "mean_rank_ic": float(metrics["rank_ic"].mean()),
            "mean_top_bottom_spread": float(metrics["top_bottom_spread"].mean()),
            "std_rank_ic": float(metrics["rank_ic"].std(ddof=0)),
        })
    return pd.DataFrame(rows)


regime_thresholds = build_regime_thresholds(final_train)
model_frame = add_regime_columns(model_frame, regime_thresholds)
feature_decisions = add_regime_columns(feature_decisions, regime_thresholds)
final_train = add_regime_columns(final_train, regime_thresholds)
test_features = add_regime_columns(test_features, regime_thresholds)
test_labeled = add_regime_columns(test_labeled, regime_thresholds)

print("Regime thresholds:")
print(pd.Series(regime_thresholds).to_string())
print("Regime counts in final train:")
print(final_train.drop_duplicates("date")["regime"].value_counts().to_string())


Regime thresholds:
high_vol          0.010106
deep_drawdown    -0.027068
weak_breadth      0.508322
strong_rebound    0.033199
Regime counts in final train:
regime
calm        175
drawdown    136
rebound      87
high_vol     17


In [8]:
# Model training helpers: LambdaRank plus residual alpha magnitude model
LABEL_GAIN = [0, 1, 2, 3, 5, 8, 13, 21, 34, 55]

RANK_PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "eval_at": [5, 10, 25],
    "label_gain": LABEL_GAIN,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 6,
    "min_data_in_leaf": 150,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.80,
    "bagging_freq": 1,
    "lambda_l1": 0.10,
    "lambda_l2": 1.00,
    "lambdarank_truncation_level": 50,
    "verbosity": -1,
}

MAG_PARAMS = {
    "objective": "regression_l1",
    "metric": "l1",
    "learning_rate": 0.02,
    "num_leaves": 31,
    "max_depth": 5,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.80,
    "bagging_freq": 1,
    "lambda_l1": 0.10,
    "lambda_l2": 1.00,
    "verbosity": -1,
}


def prepare_rank_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    ordered = frame.sort_values(["date", "ticker"]).reset_index(drop=True)
    group_sizes = ordered.groupby("date", sort=False).size().to_numpy(dtype=np.int32)
    if int(group_sizes.sum()) != len(ordered):
        raise ValueError("LightGBM group sizes do not match row count.")
    return ordered, group_sizes


def make_rank_dataset(frame: pd.DataFrame) -> tuple[lgb.Dataset, pd.DataFrame]:
    ordered, group_sizes = prepare_rank_frame(frame)
    X = ordered[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    y = ordered[LABEL_COL].astype(int)
    return lgb.Dataset(X, label=y, group=group_sizes, free_raw_data=False), ordered


def make_mag_dataset(frame: pd.DataFrame) -> tuple[lgb.Dataset, pd.DataFrame]:
    ordered = frame.sort_values(["date", "ticker"]).reset_index(drop=True)
    X = ordered[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    y = ordered["alpha_zscore"].clip(-5.0, 5.0)
    return lgb.Dataset(X, label=y, free_raw_data=False), ordered


def train_ranker(train_frame: pd.DataFrame, val_frame: pd.DataFrame | None = None, seed: int = RANDOM_SEED, num_boost_round: int = 1000):
    params = deepcopy(RANK_PARAMS)
    params.update({"seed": seed, "feature_fraction_seed": seed, "bagging_seed": seed})
    train_data, ordered_train = make_rank_dataset(train_frame)
    callbacks = [lgb.log_evaluation(period=100)]
    valid_sets = None
    valid_names = None
    ordered_val = None
    if val_frame is not None and not val_frame.empty:
        val_data, ordered_val = make_rank_dataset(val_frame)
        valid_sets = [train_data, val_data]
        valid_names = ["train", "valid"]
        callbacks.append(lgb.early_stopping(stopping_rounds=100, first_metric_only=True))
    model = lgb.train(params, train_data, num_boost_round=num_boost_round, valid_sets=valid_sets, valid_names=valid_names, callbacks=callbacks)
    return model, ordered_train, ordered_val


def train_magnitude_model(train_frame: pd.DataFrame, val_frame: pd.DataFrame | None = None, seed: int = RANDOM_SEED, num_boost_round: int = 600):
    params = deepcopy(MAG_PARAMS)
    params.update({"seed": seed, "feature_fraction_seed": seed, "bagging_seed": seed})
    train_data, ordered_train = make_mag_dataset(train_frame)
    callbacks = [lgb.log_evaluation(period=100)]
    valid_sets = None
    valid_names = None
    if val_frame is not None and not val_frame.empty:
        val_data, _ = make_mag_dataset(val_frame)
        valid_sets = [train_data, val_data]
        valid_names = ["train", "valid"]
        callbacks.append(lgb.early_stopping(stopping_rounds=100, first_metric_only=True))
    model = lgb.train(params, train_data, num_boost_round=num_boost_round, valid_sets=valid_sets, valid_names=valid_names, callbacks=callbacks)
    return model, ordered_train


def _date_median_fill(values: pd.Series, fallback: float) -> pd.Series:
    median = values[(values > 0) & values.notna()].median()
    if pd.isna(median):
        median = fallback
    return values.fillna(median).clip(lower=0.005)


def score_with_models(rank_models: list, mag_models: list, frame: pd.DataFrame, rank_weight: float = 0.75) -> pd.DataFrame:
    scoring = frame.sort_values(["date", "ticker"]).reset_index(drop=True).copy()
    X = scoring[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)

    rank_raw = np.zeros(len(scoring), dtype=float)
    for model in rank_models:
        rank_raw += model.predict(X, num_iteration=getattr(model, "best_iteration", None) or None)
    rank_raw = rank_raw / max(len(rank_models), 1)

    if mag_models:
        mag_raw = np.zeros(len(scoring), dtype=float)
        for model in mag_models:
            mag_raw += model.predict(X, num_iteration=getattr(model, "best_iteration", None) or None)
        mag_raw = mag_raw / len(mag_models)
    else:
        mag_raw = np.zeros(len(scoring), dtype=float)

    scoring["rank_model_score"] = rank_raw
    scoring["magnitude_model_score"] = mag_raw
    scoring["rank_score_pct"] = scoring.groupby("date")["rank_model_score"].rank(pct=True, method="average")
    scoring["magnitude_score_pct"] = scoring.groupby("date")["magnitude_model_score"].rank(pct=True, method="average")
    scoring["model_score"] = rank_weight * scoring["rank_score_pct"] + (1.0 - rank_weight) * scoring["magnitude_score_pct"]
    scoring["signal_rank"] = scoring.groupby("date")["model_score"].rank(pct=True, method="average")

    vol_fallback = scoring["vol_20d"].replace([np.inf, -np.inf], np.nan)
    vol_fallback = vol_fallback[(vol_fallback > 0) & vol_fallback.notna()].median()
    if pd.isna(vol_fallback):
        vol_fallback = 0.02
    scoring["expected_volatility"] = scoring.groupby("date")["vol_20d"].transform(
        lambda values: _date_median_fill(values.replace([np.inf, -np.inf], np.nan), float(vol_fallback))
    )
    scoring["expected_return"] = scoring["signal_rank"]
    scoring["horizon"] = HORIZON
    return scoring


In [36]:
# Portfolio construction and approximate validation backtest

def choose_position_count(universe_size: int, top_fraction: float, min_holdings: int, max_holdings: int) -> int:
    return int(min(max(round(universe_size * top_fraction), min_holdings), max_holdings, universe_size))


def normalize_with_cap(raw_scores: pd.Series, max_weight: float) -> pd.Series:
    scores = raw_scores.astype(float).clip(lower=0.0).replace([np.inf, -np.inf], np.nan).dropna()
    if scores.empty or scores.sum() <= 0.0:
        scores = pd.Series(1.0, index=raw_scores.index)
    weights = scores / scores.sum()
    cap = max(float(max_weight), 1.0 / len(weights))
    for _ in range(50):
        over = weights > cap
        if not over.any():
            break
        excess = float((weights.loc[over] - cap).sum())
        weights.loc[over] = cap
        under = ~over
        under_total = float(weights.loc[under].sum())
        if under_total <= 0.0 or excess <= 1e-12:
            break
        weights.loc[under] = weights.loc[under] + excess * (weights.loc[under] / under_total)
    return weights / weights.sum()


def build_beta_aware_portfolio(
    predictions: pd.DataFrame,
    *,
    dataset_name: str,
    strategy_name: str,
    config: dict,
) -> PortfolioWeights:
    validated = validate_prediction_frame(predictions, dataset_name=dataset_name, horizon=HORIZON, repo_root=repo_root)
    all_tickers = sorted(get_dataset_spec(dataset_name, repo_root=repo_root).tickers)
    rows = []
    index = []
    previous = pd.Series(0.0, index=all_tickers)

    for date_value, frame in validated.groupby("date", sort=True):
        working = frame.copy()
        stress = bool(working["stress_flag"].fillna(0.0).mean() >= 0.5) if "stress_flag" in working.columns else False
        top_fraction = config["stress_top_fraction"] if stress else config["base_top_fraction"]
        max_weight = config["stress_max_weight"] if stress else config["base_max_weight"]
        risk_power = config["stress_risk_power"] if stress else config["base_risk_power"]
        beta_penalty = config["beta_penalty"] * (1.75 if stress else 1.0)

        choose_n = choose_position_count(len(working), top_fraction, config["min_holdings"], config["max_holdings"])
        vol = working["expected_volatility"].replace([np.inf, -np.inf], np.nan)
        date_vol = vol[(vol > 0) & vol.notna()].median()
        if pd.isna(date_vol):
            date_vol = 0.02
        working["vol_for_weight"] = vol.fillna(date_vol).clip(lower=0.005)
        beta = working["beta_60d_spy"].replace([np.inf, -np.inf], np.nan).fillna(1.0)
        working["beta_penalty"] = beta_penalty * (beta - config["beta_target"]).clip(lower=0.0)
        working["adjusted_score"] = working["signal_rank"] - working["beta_penalty"]
        threshold = working["adjusted_score"].quantile(config["entry_quantile"])
        working["alpha_score"] = (working["adjusted_score"] - threshold).clip(lower=0.0)
        working["allocation_score"] = working["alpha_score"] / (working["vol_for_weight"] ** risk_power)

        selected = working.sort_values(["allocation_score", "signal_rank"], ascending=False).head(choose_n).copy()
        allocation_score = selected.set_index("ticker")["allocation_score"]
        if allocation_score.sum() <= 0.0:
            allocation_score = selected.set_index("ticker")["signal_rank"]

        selected_weights = normalize_with_cap(allocation_score, max_weight=max_weight)
        row = pd.Series(0.0, index=all_tickers)
        row.loc[selected_weights.index] = selected_weights

        turnover_blend = config["turnover_blend"]
        if turnover_blend > 0.0 and previous.sum() > 0.0:
            blended = (1.0 - turnover_blend) * row + turnover_blend * previous
            blended = blended.loc[blended > 0.0].sort_values(ascending=False).head(choose_n)
            capped = normalize_with_cap(blended, max_weight=max_weight)
            row = pd.Series(0.0, index=all_tickers)
            row.loc[capped.index] = capped

        row = row / row.sum()
        rows.append(row)
        index.append(pd.Timestamp(date_value))
        previous = row

    weights = pd.DataFrame(rows, index=pd.DatetimeIndex(index))
    weights.index.name = "date"
    weights = validate_weights_frame(weights, dataset_name=dataset_name, repo_root=repo_root)
    return PortfolioWeights(weights=weights, dataset_name=dataset_name, strategy_name=strategy_name, metadata=config)


def approximate_strategy_metrics(prices_frame: pd.DataFrame, weights: pd.DataFrame, cost_bps: float = 10.0) -> dict[str, float]:
    wide = prices_frame.pivot(index="date", columns="ticker", values="adj_close").sort_index()
    returns = wide.pct_change(fill_method=None).fillna(0.0)
    cols = [col for col in weights.columns if col in returns.columns]
    daily_weights = weights.loc[:, cols].reindex(returns.index).ffill().fillna(0.0)
    shifted_weights = daily_weights.shift(1).fillna(0.0)
    gross_returns = (shifted_weights * returns.loc[:, cols]).sum(axis=1)
    rebalance_turnover = (weights.loc[:, cols].fillna(0.0).diff().abs().sum(axis=1) / 2.0).fillna(weights.loc[:, cols].abs().sum(axis=1))
    costs = pd.Series(0.0, index=returns.index)
    aligned_turnover = rebalance_turnover.reindex(costs.index).fillna(0.0)
    costs = aligned_turnover * (float(cost_bps) / 10000.0)
    strategy_returns = (gross_returns - costs).loc[weights.index.min():weights.index.max()]
    nav = (1.0 + strategy_returns).cumprod()
    years = max((nav.index.max() - nav.index.min()).days / 365.25, 1e-9)
    total_return = float(nav.iloc[-1] - 1.0)
    annual_return = float(nav.iloc[-1] ** (1.0 / years) - 1.0)
    annual_vol = float(strategy_returns.std(ddof=0) * np.sqrt(252.0))
    sharpe = float(annual_return / annual_vol) if annual_vol > 0 else np.nan
    drawdown = nav / nav.cummax() - 1.0
    average_turnover = float(rebalance_turnover.mean())
    return {
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "max_drawdown": float(drawdown.min()),
        "average_turnover": average_turnover,
    }


BASE_PORTFOLIO_CONFIG = {
    "base_top_fraction": BASE_TOP_FRACTION,
    "stress_top_fraction": STRESS_TOP_FRACTION,
    "min_holdings": MIN_HOLDINGS,
    "max_holdings": MAX_HOLDINGS,
    "base_max_weight": BASE_MAX_WEIGHT,
    "stress_max_weight": STRESS_MAX_WEIGHT,
    "turnover_blend": TURNOVER_BLEND,
    "base_risk_power": BASE_RISK_POWER,
    "stress_risk_power": STRESS_RISK_POWER,
    "beta_target": BETA_TARGET,
    "beta_penalty": BETA_PENALTY,
    "entry_quantile": ENTRY_QUANTILE,
}


In [37]:
# V4 upgrade: core-satellite portfolio layer.
# Defensive core = inverse-vol / beta-aware weights.
# Alpha sleeve = LightGBM rank-selected weights.
# In stress regimes, reduce alpha sleeve and lean harder on defensive core.

def build_beta_aware_portfolio(
    predictions: pd.DataFrame,
    *,
    dataset_name: str,
    strategy_name: str,
    config: dict,
) -> PortfolioWeights:
    validated = validate_prediction_frame(predictions, dataset_name=dataset_name, horizon=HORIZON, repo_root=repo_root)
    all_tickers = sorted(get_dataset_spec(dataset_name, repo_root=repo_root).tickers)

    rows = []
    index = []
    previous = pd.Series(0.0, index=all_tickers)

    for date_value, frame in validated.groupby("date", sort=True):
        working = frame.copy()

        stress = bool(working["stress_flag"].fillna(0.0).mean() >= 0.5) if "stress_flag" in working.columns else False
        top_fraction = config["stress_top_fraction"] if stress else config["base_top_fraction"]
        max_weight = config["stress_max_weight"] if stress else config["base_max_weight"]
        risk_power = config["stress_risk_power"] if stress else config["base_risk_power"]
        beta_penalty = config["beta_penalty"] * (1.75 if stress else 1.0)
        alpha_tilt = config["stress_alpha_tilt"] if stress else config["base_alpha_tilt"]

        vol = working["expected_volatility"].replace([np.inf, -np.inf], np.nan)
        date_vol = vol[(vol > 0) & vol.notna()].median()
        if pd.isna(date_vol):
            date_vol = 0.02

        working["vol_for_weight"] = vol.fillna(date_vol).clip(lower=0.005)
        beta = working["beta_60d_spy"].replace([np.inf, -np.inf], np.nan).fillna(1.0)
        beta_excess = (beta - config["beta_target"]).clip(lower=0.0)

        # Defensive core: own the whole available universe, but penalize volatility and high beta.
        working["core_score"] = 1.0 / (working["vol_for_weight"] * (1.0 + config["core_beta_penalty"] * beta_excess))
        core_weights = normalize_with_cap(
            working.set_index("ticker")["core_score"],
            max_weight=max_weight,
        )

        # Alpha sleeve: model-selected names with beta and volatility penalties.
        choose_n = choose_position_count(len(working), top_fraction, config["min_holdings"], config["max_holdings"])
        working["beta_penalty"] = beta_penalty * beta_excess
        working["adjusted_score"] = working["signal_rank"] - working["beta_penalty"]

        threshold = working["adjusted_score"].quantile(config["entry_quantile"])
        working["alpha_score"] = (working["adjusted_score"] - threshold).clip(lower=0.0)
        working["allocation_score"] = working["alpha_score"] / (working["vol_for_weight"] ** risk_power)

        selected = working.sort_values(["allocation_score", "signal_rank"], ascending=False).head(choose_n).copy()
        sleeve_score = selected.set_index("ticker")["allocation_score"]
        if sleeve_score.sum() <= 0.0:
            sleeve_score = selected.set_index("ticker")["signal_rank"]

        sleeve_weights = normalize_with_cap(sleeve_score, max_weight=max_weight)

        # Dynamic confidence boost: if the model has a wider cross-sectional spread, let it matter more.
        score_spread = working["model_score"].quantile(0.90) - working["model_score"].quantile(0.50)
        if pd.notna(score_spread) and score_spread > config["confidence_spread_threshold"]:
            alpha_tilt = min(alpha_tilt + config["confidence_tilt_boost"], config["max_alpha_tilt"])

        row = pd.Series(0.0, index=all_tickers)
        row.loc[core_weights.index] += (1.0 - alpha_tilt) * core_weights
        row.loc[sleeve_weights.index] += alpha_tilt * sleeve_weights

        if config["turnover_blend"] > 0.0 and previous.sum() > 0.0:
            row = (1.0 - config["turnover_blend"]) * row + config["turnover_blend"] * previous

        positive = row.loc[row > 0.0]
        capped = normalize_with_cap(positive, max_weight=max_weight)

        row = pd.Series(0.0, index=all_tickers)
        row.loc[capped.index] = capped
        row = row / row.sum()

        rows.append(row)
        index.append(pd.Timestamp(date_value))
        previous = row

    weights = pd.DataFrame(rows, index=pd.DatetimeIndex(index))
    weights.index.name = "date"
    weights = validate_weights_frame(weights, dataset_name=dataset_name, repo_root=repo_root)

    return PortfolioWeights(
        weights=weights,
        dataset_name=dataset_name,
        strategy_name=strategy_name,
        metadata=config,
    )


BASE_PORTFOLIO_CONFIG = {
    "base_top_fraction": BASE_TOP_FRACTION,
    "stress_top_fraction": STRESS_TOP_FRACTION,
    "min_holdings": MIN_HOLDINGS,
    "max_holdings": MAX_HOLDINGS,
    "base_max_weight": BASE_MAX_WEIGHT,
    "stress_max_weight": STRESS_MAX_WEIGHT,
    "turnover_blend": TURNOVER_BLEND,
    "base_risk_power": BASE_RISK_POWER,
    "stress_risk_power": STRESS_RISK_POWER,
    "beta_target": BETA_TARGET,
    "beta_penalty": BETA_PENALTY,
    "entry_quantile": ENTRY_QUANTILE,
    "base_alpha_tilt": 0.30,
    "stress_alpha_tilt": 0.12,
    "max_alpha_tilt": 0.45,
    "confidence_tilt_boost": 0.10,
    "confidence_spread_threshold": 0.15,
    "core_beta_penalty": 0.75,
}


In [38]:
# Walk-forward CV: train models and collect out-of-sample validation predictions
fold_rows = []
fold_best_iterations = []
fold_rank_models = []
validation_predictions = []
fold_importance_frames = []

for fold in walk_forward_folds:
    print("Training", fold["name"])

    # Folds were created before regime tagging, so add regime columns here.
    train_fold = add_regime_columns(fold["train"], regime_thresholds)
    val_fold = add_regime_columns(fold["val"], regime_thresholds)

    rank_model, _, ordered_val = train_ranker(train_fold, val_fold, seed=RANDOM_SEED, num_boost_round=1000)
    mag_model, _ = train_magnitude_model(train_fold, val_fold, seed=RANDOM_SEED, num_boost_round=600)

    best_iteration = int(rank_model.best_iteration or rank_model.current_iteration())
    fold_best_iterations.append(best_iteration)

    ordered_val = add_regime_columns(ordered_val, regime_thresholds)
    scored_val = score_with_models([rank_model], [mag_model], ordered_val)

    target_cols = ["date", "ticker", TARGET_COL, "alpha_rank_pct", "alpha_zscore"]
    scored_val = scored_val.merge(
        ordered_val[target_cols],
        on=["date", "ticker"],
        how="left",
        suffixes=("", "_target"),
    )

    validation_predictions.append(scored_val)

    all_diag = score_diagnostics(scored_val)
    row = {
        "fold": fold["name"],
        "best_iteration": best_iteration,
        "train_rows": len(train_fold),
        "val_rows": len(val_fold),
    }

    if not all_diag.empty:
        row.update(all_diag.iloc[0].drop(labels=["group"]).to_dict())

    fold_rows.append(row)

    fold_importance_frames.append(pd.DataFrame({
        "fold": fold["name"],
        "feature": FEATURE_COLS,
        "gain": rank_model.feature_importance(importance_type="gain"),
        "split": rank_model.feature_importance(importance_type="split"),
    }))

cv_results = pd.DataFrame(fold_rows)
oos_validation_predictions = pd.concat(validation_predictions, ignore_index=True) if validation_predictions else pd.DataFrame()

print("Walk-forward model diagnostics:")
print(cv_results.to_string(index=False))

print("Validation regime diagnostics:")
print(score_diagnostics(oos_validation_predictions, group_col="regime").to_string(index=False))


Training wf_2018
Training until validation scores don't improve for 100 rounds
[100]	train's ndcg@5: 0.751083	train's ndcg@10: 0.667154	train's ndcg@25: 0.557894	valid's ndcg@5: 0.38948	valid's ndcg@10: 0.38746	valid's ndcg@25: 0.364974
Early stopping, best iteration is:
[9]	train's ndcg@5: 0.556448	train's ndcg@10: 0.520035	train's ndcg@25: 0.467769	valid's ndcg@5: 0.43429	valid's ndcg@10: 0.405429	valid's ndcg@25: 0.376743
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.691852	valid's l1: 0.719576
Early stopping, best iteration is:
[38]	train's l1: 0.697564	valid's l1: 0.718704
Evaluated only: l1
Training wf_2019
Training until validation scores don't improve for 100 rounds
[100]	train's ndcg@5: 0.713351	train's ndcg@10: 0.631232	train's ndcg@25: 0.532902	valid's ndcg@5: 0.464205	valid's ndcg@10: 0.420049	valid's ndcg@25: 0.382636
[200]	train's ndcg@5: 0.815183	train's ndcg@10: 0.717779	train's ndcg@25: 0.590176	valid's ndcg@5:

In [39]:
# Validation-only portfolio sweep. This locks settings before test.
portfolio_grid = []

for turnover_blend in [0.65, 0.80, 0.90]:
    for beta_penalty in [0.08, 0.14, 0.20]:
        for base_alpha_tilt in [0.20, 0.30, 0.40]:
            for stress_alpha_tilt in [0.05, 0.10, 0.15]:
                config = deepcopy(BASE_PORTFOLIO_CONFIG)
                config.update({
                    "base_top_fraction": 0.45,
                    "stress_top_fraction": 0.60,
                    "turnover_blend": turnover_blend,
                    "beta_penalty": beta_penalty,
                    "base_risk_power": 0.25,
                    "stress_risk_power": 0.60,
                    "base_alpha_tilt": base_alpha_tilt,
                    "stress_alpha_tilt": stress_alpha_tilt,
                    "core_beta_penalty": 1.00,
                })
                portfolio_grid.append(config)

sweep_rows = []

for idx, config in enumerate(portfolio_grid):
    try:
        portfolio_candidate = build_beta_aware_portfolio(
            oos_validation_predictions,
            dataset_name=DATASET_NAME,
            strategy_name=f"v4_validation_candidate_{idx}",
            config=config,
        )
        metrics = approximate_strategy_metrics(prices, portfolio_candidate.weights, cost_bps=spec.cost_bps)

        # More defensive selection score for this week's hostile hidden data.
        selection_score = (
            metrics["sharpe"]
            - 0.30 * metrics["average_turnover"]
            + 0.85 * metrics["max_drawdown"]
            + 0.20 * metrics["annual_return"]
        )

        sweep_rows.append({
            "candidate": idx,
            "selection_score": selection_score,
            **metrics,
            **config,
        })
    except Exception as exc:
        print("Skipped candidate", idx, exc)

sweep_results = pd.DataFrame(sweep_rows).sort_values("selection_score", ascending=False).reset_index(drop=True)

if sweep_results.empty:
    locked_portfolio_config = deepcopy(BASE_PORTFOLIO_CONFIG)
    print("Portfolio sweep failed. Falling back to base config.")
else:
    config_cols = list(BASE_PORTFOLIO_CONFIG.keys())
    locked_portfolio_config = sweep_results.loc[0, config_cols].to_dict()

    for key in ["min_holdings", "max_holdings"]:
        locked_portfolio_config[key] = int(locked_portfolio_config[key])

    print("Top validation portfolio settings:")
    print(
        sweep_results.head(10)[
            [
                "candidate",
                "selection_score",
                "annual_return",
                "sharpe",
                "max_drawdown",
                "average_turnover",
                *config_cols,
            ]
        ].to_string(index=False)
    )

print("Locked portfolio config:")
print(pd.Series(locked_portfolio_config).to_string())


Top validation portfolio settings:
 candidate  selection_score  annual_return   sharpe  max_drawdown  average_turnover  base_top_fraction  stress_top_fraction  min_holdings  max_holdings  base_max_weight  stress_max_weight  turnover_blend  base_risk_power  stress_risk_power  beta_target  beta_penalty  entry_quantile  base_alpha_tilt  stress_alpha_tilt  max_alpha_tilt  confidence_tilt_boost  confidence_spread_threshold  core_beta_penalty
        62         0.689356       0.212133 0.969874     -0.373892          0.017124               0.45                  0.6            50           180            0.025               0.02             0.9             0.25                0.6         1.05          0.08             0.5              0.4               0.15            0.45                    0.1                         0.15                1.0
        35         0.688276       0.213329 0.973948     -0.375117          0.031627               0.45                  0.6            50           180  

In [40]:
# Final seed ensemble trained after validation settings are locked
if fold_best_iterations:
    FINAL_NUM_BOOST_ROUND = int(np.nanmedian(fold_best_iterations))
else:
    FINAL_NUM_BOOST_ROUND = 80
FINAL_NUM_BOOST_ROUND = int(max(min(FINAL_NUM_BOOST_ROUND, 160), 30))
MAG_NUM_BOOST_ROUND = 200

print("Final train rows:", len(final_train))
print("Final train dates:", final_train["date"].nunique())
print("Final rank boosting rounds:", FINAL_NUM_BOOST_ROUND)
print("Final magnitude boosting rounds:", MAG_NUM_BOOST_ROUND)

rank_models = []
mag_models = []
for seed in ENSEMBLE_SEEDS:
    print("Final ensemble seed", seed)
    rank_model, _, _ = train_ranker(final_train, None, seed=seed, num_boost_round=FINAL_NUM_BOOST_ROUND)
    mag_model, _ = train_magnitude_model(final_train, None, seed=seed, num_boost_round=MAG_NUM_BOOST_ROUND)
    rank_models.append(rank_model)
    mag_models.append(mag_model)

print("Trained rank models:", len(rank_models))
print("Trained magnitude models:", len(mag_models))


Final train rows: 197658
Final train dates: 415
Final rank boosting rounds: 57
Final magnitude boosting rounds: 200
Final ensemble seed 17
Final ensemble seed 42
Final ensemble seed 101
Trained rank models: 3
Trained magnitude models: 3


In [41]:
# Locked test predictions and portfolio
predictions_full = score_with_models(rank_models, mag_models, test_features)
predictions = predictions_full[[
    "date", "ticker", "horizon", "expected_return", "expected_volatility",
    "signal_date", "model_score", "signal_rank", "rank_model_score", "magnitude_model_score",
    "beta_60d_spy", "regime", "stress_flag",
]].copy()
predictions = validate_prediction_frame(predictions, dataset_name=DATASET_NAME, horizon=HORIZON, repo_root=repo_root)

scored_test_labeled = predictions_full.merge(
    test_labeled[["date", "ticker", TARGET_COL, "alpha_rank_pct", "alpha_zscore", "regime"]],
    on=["date", "ticker"],
    how="left",
    suffixes=("", "_target"),
)
if "regime_target" in scored_test_labeled.columns:
    scored_test_labeled["regime"] = scored_test_labeled["regime_target"].fillna(scored_test_labeled["regime"])
    scored_test_labeled = scored_test_labeled.drop(columns=["regime_target"])

test_signal_metrics = score_diagnostics(scored_test_labeled)
test_regime_metrics = score_diagnostics(scored_test_labeled, group_col="regime")

portfolio = build_beta_aware_portfolio(
    predictions,
    dataset_name=DATASET_NAME,
    strategy_name=STRATEGY_NAME,
    config=locked_portfolio_config,
)
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=DATASET_NAME, repo_root=repo_root)

print("Predictions:", predictions.shape)
print("Weights:", validated_weights.shape)
print("Average names held:", (validated_weights > 0).sum(axis=1).mean())
print("Max single-name weight:", validated_weights.max(axis=1).max())
print("Test signal metrics:")
print(test_signal_metrics.to_string(index=False))
print("Test regime metrics:")
print(test_regime_metrics.to_string(index=False))


Predictions: (25791, 13)
Weights: (52, 503)
Average names held: 495.9807692307692
Max single-name weight: 0.006983504401265581
Test signal metrics:
group  dates   mean_ic  mean_rank_ic  mean_top_bottom_spread  std_rank_ic
  all     52 -0.011003     -0.006778               -0.001157     0.229153
Test regime metrics:
   group  dates   mean_ic  mean_rank_ic  mean_top_bottom_spread  std_rank_ic
    calm      1 -0.382266     -0.425609               -0.061642     0.000000
drawdown     45  0.003665      0.009862                0.001254     0.235665
 rebound      6 -0.059137     -0.061769               -0.009154     0.064974


In [42]:
# Final local test backtest and baselines.
# Important: do not use toolkit backtest_weights here after narrowing TEST_END to 2022.
# backtest_weights loads the full cached dataset and would hold the last 2022 weights through 2025.

def metrics_from_returns(returns: pd.Series) -> dict[str, float]:
    returns = returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    nav = (1.0 + returns).cumprod()
    years = max((nav.index.max() - nav.index.min()).days / 365.25, 1e-9)
    total_return = float(nav.iloc[-1] - 1.0)
    annual_return = float(nav.iloc[-1] ** (1.0 / years) - 1.0)
    annual_volatility = float(returns.std(ddof=0) * np.sqrt(252.0))
    sharpe = float(annual_return / annual_volatility) if annual_volatility > 0 else np.nan
    downside = returns.clip(upper=0.0)
    downside_volatility = float(downside.std(ddof=0) * np.sqrt(252.0))
    sortino = float(annual_return / downside_volatility) if downside_volatility > 0 else np.nan
    drawdown = nav / nav.cummax() - 1.0

    return {
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_volatility,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": float(drawdown.min()),
    }


def local_strategy_returns(
    prices_frame: pd.DataFrame,
    weights: pd.DataFrame,
    *,
    start_date,
    end_date,
    cost_bps: float,
) -> tuple[pd.Series, pd.Series]:
    wide = prices_frame.pivot(index="date", columns="ticker", values="adj_close").sort_index()
    returns = wide.pct_change(fill_method=None).fillna(0.0)

    cols = [col for col in weights.columns if col in returns.columns]
    evaluation_index = returns.loc[pd.Timestamp(start_date):pd.Timestamp(end_date)].index

    daily_weights = weights.loc[:, cols].reindex(evaluation_index).ffill().fillna(0.0)
    shifted_weights = daily_weights.shift(1).fillna(0.0)
    gross_returns = (shifted_weights * returns.loc[evaluation_index, cols]).sum(axis=1)

    rebalance_turnover = (
        weights.loc[:, cols]
        .fillna(0.0)
        .diff()
        .abs()
        .sum(axis=1)
        / 2.0
    )
    if not rebalance_turnover.empty:
        rebalance_turnover.iloc[0] = float(weights.loc[:, cols].iloc[0].abs().sum())

    daily_costs = rebalance_turnover.reindex(evaluation_index).fillna(0.0) * (float(cost_bps) / 10000.0)
    net_returns = (gross_returns - daily_costs).rename("strategy_return")
    return net_returns, rebalance_turnover


def equal_weight_like(weights: pd.DataFrame) -> pd.DataFrame:
    cols = list(weights.columns)
    return pd.DataFrame(
        1.0 / len(cols),
        index=weights.index,
        columns=cols,
    )


def inverse_vol_from_predictions(predictions: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    index = []

    for date_value, frame in predictions.groupby("date", sort=True):
        vol = frame.set_index("ticker")["expected_volatility"].replace([np.inf, -np.inf], np.nan)
        vol = vol.reindex(columns)
        fallback = vol[(vol > 0) & vol.notna()].median()
        if pd.isna(fallback):
            fallback = 0.02

        inv = 1.0 / vol.fillna(fallback).clip(lower=0.005)
        inv = inv / inv.sum()

        rows.append(inv)
        index.append(pd.Timestamp(date_value))

    weights = pd.DataFrame(rows, index=pd.DatetimeIndex(index), columns=columns)
    weights.index.name = "date"
    return validate_weights_frame(weights, dataset_name=DATASET_NAME, repo_root=repo_root)


strategy_returns, strategy_turnover = local_strategy_returns(
    prices,
    validated_weights,
    start_date=TEST_START,
    end_date=TEST_END,
    cost_bps=spec.cost_bps,
)

benchmark_prices = (
    prices.loc[prices["ticker"] == spec.benchmark_ticker, ["date", "adj_close"]]
    .drop_duplicates("date")
    .set_index("date")["adj_close"]
    .sort_index()
)
benchmark_returns = (
    benchmark_prices
    .pct_change(fill_method=None)
    .fillna(0.0)
    .loc[pd.Timestamp(TEST_START):pd.Timestamp(TEST_END)]
    .rename("benchmark_return")
)

equal_weights = equal_weight_like(validated_weights)
equal_returns, equal_turnover = local_strategy_returns(
    prices,
    equal_weights,
    start_date=TEST_START,
    end_date=TEST_END,
    cost_bps=spec.cost_bps,
)

inverse_vol_weights = inverse_vol_from_predictions(predictions, list(validated_weights.columns))
inverse_vol_returns, inverse_vol_turnover = local_strategy_returns(
    prices,
    inverse_vol_weights,
    start_date=TEST_START,
    end_date=TEST_END,
    cost_bps=spec.cost_bps,
)

strategy_metrics = metrics_from_returns(strategy_returns)
benchmark_metrics = metrics_from_returns(benchmark_returns)
equal_metrics = metrics_from_returns(equal_returns)
inverse_vol_metrics = metrics_from_returns(inverse_vol_returns)

strategy_metrics["average_turnover"] = float(strategy_turnover.mean())
benchmark_metrics["average_turnover"] = np.nan
equal_metrics["average_turnover"] = float(equal_turnover.mean())
inverse_vol_metrics["average_turnover"] = float(inverse_vol_turnover.mean())

result = type("LocalBacktestResult", (), {})()
result.metrics = {
    **strategy_metrics,
    "benchmark_annual_return": benchmark_metrics["annual_return"],
    "benchmark_sharpe": benchmark_metrics["sharpe"],
    "benchmark_max_drawdown": benchmark_metrics["max_drawdown"],
    "sharpe_vs_benchmark": strategy_metrics["sharpe"] - benchmark_metrics["sharpe"],
}
artifacts = {}

print("Brian LightGBM v4 local test window")
print("Test window:", TEST_START.date(), "to", TEST_END.date())
for key in [
    "annual_return",
    "annual_volatility",
    "sharpe",
    "sortino",
    "max_drawdown",
    "average_turnover",
    "benchmark_annual_return",
    "benchmark_sharpe",
    "benchmark_max_drawdown",
    "sharpe_vs_benchmark",
]:
    print(f"{key}: {result.metrics.get(key)}")

comparison = pd.DataFrame([
    {
        "strategy": STRATEGY_NAME,
        "annual_return": strategy_metrics["annual_return"],
        "sharpe": strategy_metrics["sharpe"],
        "max_drawdown": strategy_metrics["max_drawdown"],
        "average_turnover": strategy_metrics["average_turnover"],
    },
    {
        "strategy": spec.benchmark_ticker,
        "annual_return": benchmark_metrics["annual_return"],
        "sharpe": benchmark_metrics["sharpe"],
        "max_drawdown": benchmark_metrics["max_drawdown"],
        "average_turnover": np.nan,
    },
    {
        "strategy": "equal_weight_local",
        "annual_return": equal_metrics["annual_return"],
        "sharpe": equal_metrics["sharpe"],
        "max_drawdown": equal_metrics["max_drawdown"],
        "average_turnover": equal_metrics["average_turnover"],
    },
    {
        "strategy": "inverse_vol_local",
        "annual_return": inverse_vol_metrics["annual_return"],
        "sharpe": inverse_vol_metrics["sharpe"],
        "max_drawdown": inverse_vol_metrics["max_drawdown"],
        "average_turnover": inverse_vol_metrics["average_turnover"],
    },
]).sort_values("sharpe", ascending=False)

print("Baseline comparison:")
print(comparison.to_string(index=False))


Brian LightGBM v4 local test window
Test window: 2022-01-03 to 2022-12-31
annual_return: -0.09050231712855616
annual_volatility: 0.23308722969764192
sharpe: -0.38827660033522526
sortino: -0.668070456789813
max_drawdown: -0.19144211892397422
average_turnover: 0.03249009507721022
benchmark_annual_return: -0.18368378895025694
benchmark_sharpe: -0.7591923968011659
benchmark_max_drawdown: -0.24496387687011845
sharpe_vs_benchmark: 0.37091579646594064
Baseline comparison:
                            strategy  annual_return    sharpe  max_drawdown  average_turnover
                   inverse_vol_local      -0.080053 -0.378018     -0.177788          0.069131
brian_lgbm_v4_regime_beta_controlled      -0.090502 -0.388277     -0.191442          0.032490
                  equal_weight_local      -0.098035 -0.417610     -0.196557          0.019231
                                 SPY      -0.183684 -0.759192     -0.244964               NaN


In [43]:
# Crash-day diagnostics and ablation table

def compute_strategy_daily_returns(prices_frame: pd.DataFrame, weights: pd.DataFrame) -> pd.Series:
    wide = prices_frame.pivot(index="date", columns="ticker", values="adj_close").sort_index()
    returns = wide.pct_change(fill_method=None).fillna(0.0)
    cols = [col for col in weights.columns if col in returns.columns]
    daily_weights = weights.loc[:, cols].reindex(returns.index).ffill().fillna(0.0)
    return (daily_weights.shift(1).fillna(0.0) * returns.loc[:, cols]).sum(axis=1).rename("strategy_return")


strategy_daily_returns = compute_strategy_daily_returns(prices, validated_weights)
benchmark_prices = prices.loc[prices["ticker"] == spec.benchmark_ticker, ["date", "adj_close"]].drop_duplicates("date").set_index("date")["adj_close"].sort_index()
benchmark_daily_returns = benchmark_prices.pct_change().fillna(0.0).rename("benchmark_return")
crash_dates = benchmark_daily_returns.loc[validated_weights.index.min():validated_weights.index.max()].nsmallest(10).index

crash_rows = []
for date_value in crash_dates:
    nearest_weight_date = validated_weights.index[validated_weights.index.searchsorted(date_value, side="right") - 1]
    row = validated_weights.loc[nearest_weight_date]
    active = row[row > 0]
    crash_rows.append({
        "date": date_value,
        "benchmark_return": benchmark_daily_returns.loc[date_value],
        "strategy_return": strategy_daily_returns.reindex([date_value]).iloc[0],
        "active_names": int((row > 0).sum()),
        "max_weight": float(row.max()),
        "weight_date": nearest_weight_date,
    })
crash_diagnostics = pd.DataFrame(crash_rows)
print("Worst benchmark days diagnostic:")
print(crash_diagnostics.to_string(index=False))

ablation_configs = {
    "full_v4": locked_portfolio_config,
    "no_beta_penalty": {**locked_portfolio_config, "beta_penalty": 0.0},
    "low_turnover": {**locked_portfolio_config, "turnover_blend": min(0.85, locked_portfolio_config["turnover_blend"] + 0.15)},
    "more_concentrated": {**locked_portfolio_config, "base_top_fraction": max(0.25, locked_portfolio_config["base_top_fraction"] - 0.15), "stress_top_fraction": max(0.35, locked_portfolio_config["stress_top_fraction"] - 0.15)},
}
ablation_rows = []
for name, cfg in ablation_configs.items():
    try:
        ab_port = build_beta_aware_portfolio(predictions, dataset_name=DATASET_NAME, strategy_name=f"ablation_{name}", config=cfg)
        metrics = approximate_strategy_metrics(prices, ab_port.weights, cost_bps=spec.cost_bps)
        ablation_rows.append({"ablation": name, **metrics})
    except Exception as exc:
        ablation_rows.append({"ablation": name, "error": str(exc)})
ablation_table = pd.DataFrame(ablation_rows).sort_values("sharpe", ascending=False, na_position="last")
print("Ablation table using approximate test-window evaluator:")
print(ablation_table.to_string(index=False))


Worst benchmark days diagnostic:
      date  benchmark_return  strategy_return  active_names  max_weight weight_date
2022-09-13         -0.043483        -0.038006           496    0.003558  2022-09-12
2022-05-18         -0.040312        -0.038260           496    0.003400  2022-05-16
2022-06-13         -0.037968        -0.041024           496    0.003427  2022-06-13
2022-04-29         -0.036956        -0.031129           496    0.003654  2022-04-25
2022-05-05         -0.035543        -0.031301           496    0.003496  2022-05-02
2022-08-26         -0.033849        -0.030477           496    0.003731  2022-08-22
2022-06-16         -0.033096        -0.035169           496    0.003427  2022-06-13
2022-05-09         -0.032017        -0.031850           496    0.003381  2022-05-09
2022-03-07         -0.029479        -0.030100           496    0.004414  2022-03-07
2022-06-10         -0.028996        -0.025802           496    0.003270  2022-06-06
Ablation table using approximate test-windo

In [44]:
# Feature importance with fold stability
final_importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "final_gain_mean": np.mean([model.feature_importance(importance_type="gain") for model in rank_models], axis=0),
    "final_split_mean": np.mean([model.feature_importance(importance_type="split") for model in rank_models], axis=0),
})
if fold_importance_frames:
    fold_importance = pd.concat(fold_importance_frames, ignore_index=True)
    stability = (
        fold_importance
        .groupby("feature")
        .agg(fold_gain_mean=("gain", "mean"), fold_gain_std=("gain", "std"), fold_split_mean=("split", "mean"))
        .reset_index()
    )
    importance = final_importance.merge(stability, on="feature", how="left")
else:
    importance = final_importance
importance = importance.sort_values("final_gain_mean", ascending=False)
print("Top 25 features:")
print(importance.head(25).to_string(index=False))


Top 25 features:
                     feature  final_gain_mean  final_split_mean  fold_gain_mean  fold_gain_std  fold_split_mean
             cs_rank_vol_60d      3129.117346         68.333333     1649.682708    1043.840666            73.25
                cs_z_vol_60d      1344.940716         38.000000     1263.308194     607.014193            61.50
       return_dispersion_20d       406.814556        123.000000      211.055444     117.534759            92.00
          cs_z_momentum_120d       314.888873         69.666667      230.841136     137.875023            69.75
              spy_return_60d       302.073976         94.666667      154.137096      80.342744            51.75
        mom_60_20_divergence       248.510398         37.333333       56.445223      48.867648            23.50
       cs_rank_momentum_120d       240.190031         57.666667      107.205459      65.568547            39.75
                cs_z_vol_20d       187.708664         23.000000       68.239729      46

In [45]:
# Required submission function
# build_model_features(prices) is defined above and does not use targets.

submission_model_bundle = {
    "rank_models": rank_models,
    "magnitude_models": mag_models,
    "rank_weight": 0.75,
    "portfolio_config": locked_portfolio_config,
}


def predict_from_prices(model, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    if isinstance(model, dict):
        rank_model_list = model.get("rank_models", [])
        mag_model_list = model.get("magnitude_models", [])
        rank_weight = float(model.get("rank_weight", 0.75))
    else:
        rank_model_list = [model]
        mag_model_list = []
        rank_weight = 1.0

    features_local = build_model_features(prices)
    calendar_local = build_decision_calendar(prices)
    scoring_frame = features_local.rename(columns={"date": "signal_date"}).merge(calendar_local, on="signal_date", how="inner")
    scoring_frame = scoring_frame[scoring_frame["ticker"].str.upper() != "SPY"].copy()
    scoring_frame = add_regime_columns(scoring_frame, regime_thresholds)

    if dates is not None:
        requested_dates = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates), utc=True).dt.tz_localize(None).unique())
        scoring_frame = scoring_frame.loc[scoring_frame["date"].isin(requested_dates)].copy()
    if tickers is not None:
        requested_tickers = [str(ticker).upper() for ticker in tickers]
        scoring_frame = scoring_frame.loc[scoring_frame["ticker"].isin(requested_tickers)].copy()

    scored = score_with_models(rank_model_list, mag_model_list, scoring_frame, rank_weight=rank_weight)
    predictions = scored[[
        "date", "ticker", "horizon", "expected_return", "expected_volatility",
        "signal_date", "model_score", "signal_rank", "rank_model_score", "magnitude_model_score",
        "beta_60d_spy", "regime", "stress_flag",
    ]].copy()
    return validate_prediction_frame(predictions, horizon=HORIZON)


submission_check = predict_from_prices(submission_model_bundle, prices, dates=validated_weights.index, tickers=spec.tickers)
print("Submission prediction check:", submission_check.shape)
print(submission_check.head().to_string(index=False))


Submission prediction check: (25791, 13)
      date ticker  horizon  expected_return  expected_volatility signal_date  model_score  signal_rank  rank_model_score  magnitude_model_score  beta_60d_spy  regime  stress_flag
2022-01-03      A       10         0.279798             0.013501  2021-12-31     0.311616     0.279798         -0.390679              -0.020628      0.985629 rebound          0.0
2022-01-03   AAPL       10         0.133333             0.018701  2021-12-31     0.185354     0.133333         -0.400879              -0.048074      1.092697 rebound          0.0
2022-01-03   ABBV       10         0.008081             0.009572  2021-12-31     0.030808     0.008081         -0.407107              -0.155506      0.409249 rebound          0.0
2022-01-03   ABNB       10         0.989899             0.032233  2021-12-31     0.985859     0.989899          0.488474               0.085610      2.127114 rebound          0.0
2022-01-03    ABT       10         0.036364             0.011641

In [51]:
# Optional local artifact save and optional MLflow logging
metadata = {
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "dataset": DATASET_NAME,
    "horizon": HORIZON,
    "target": f"{TARGET_COL}_rank_and_alpha_zscore",
    "feature_count": len(FEATURE_COLS),
    "features": FEATURE_COLS,
    "rank_params": RANK_PARAMS,
    "magnitude_params": MAG_PARAMS,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "final_num_boost_round": FINAL_NUM_BOOST_ROUND,
    "locked_portfolio_config": locked_portfolio_config,
    "regime_thresholds": regime_thresholds,
    "cv_results": cv_results.to_dict(orient="records"),
    "portfolio_sweep_top10": sweep_results.head(10).to_dict(orient="records") if not sweep_results.empty else [],
    "test_signal_metrics": test_signal_metrics.to_dict(orient="records") if not test_signal_metrics.empty else [],
    "backtest_metrics": result.metrics,
    "notes": "V4 uses validation-only portfolio selection, LambdaRank plus alpha-zscore ensemble, regime diagnostics, and beta-aware dynamic portfolio controls.",
}

saved_model_paths = []
metadata_path = model_dir / "lgbm_v4_metadata.json"
if SAVE_LOCAL_ARTIFACTS:
    model_dir.mkdir(parents=True, exist_ok=True)
    for idx, model in enumerate(rank_models):
        path = model_dir / f"lgbm_v4_rank_seed_{ENSEMBLE_SEEDS[idx]}.txt"
        model.save_model(str(path))
        saved_model_paths.append(path)
    for idx, model in enumerate(mag_models):
        path = model_dir / f"lgbm_v4_magnitude_seed_{ENSEMBLE_SEEDS[idx]}.txt"
        model.save_model(str(path))
        saved_model_paths.append(path)
    metadata_path.write_text(json.dumps(metadata, indent=2, default=str) + "\n", encoding="utf-8")
    print("Saved local artifacts to", model_dir)
else:
    print("SAVE_LOCAL_ARTIFACTS is False. No model files were written for this notebook-only PR.")

if RUN_MLFLOW:
    import mlflow
    init_mlflow(repo_root=repo_root)
    with start_run(
        run_name=MODEL_NAME,
        dataset_name=DATASET_NAME,
        tags={
            "model_type": "lightgbm",
            "model_family": "lightgbm",
            "version": "4",
            "owner": "Brian",
            "horizon": str(HORIZON),
            "strategy_type": "regime_beta_controlled_ranker",
        },
        repo_root=repo_root,
    ):
        mlflow.log_params({
            "model_name": MODEL_NAME,
            "horizon": HORIZON,
            "feature_count": len(FEATURE_COLS),
            "rank_objective": RANK_PARAMS["objective"],
            "magnitude_objective": MAG_PARAMS["objective"],
            "ensemble_size": len(rank_models),
            "portfolio_selection": "validation_sweep",
        })
        log_predictions(predictions)
        log_portfolio(portfolio)
        log_backtest(result)
        if SAVE_LOCAL_ARTIFACTS and saved_model_paths:
            log_model_submission(
                {path.stem: path for path in saved_model_paths + [metadata_path]},
                model_name=MODEL_NAME,
                model_family="lightgbm",
                feature_names=FEATURE_COLS,
                target=f"{TARGET_COL}_rank_and_alpha_zscore",
                horizon=HORIZON,
                rebalance_frequency=REBALANCE_FREQUENCY,
                preprocessing={"missing_values": "left_as_nan", "portfolio_selection": "validation_only"},
                model_config=metadata,
                source_files=[notebook_path],
                notes=metadata["notes"],
            )
else:
    print("RUN_MLFLOW is False. Set True only when intentionally logging.")


SAVE_LOCAL_ARTIFACTS is False. No model files were written for this notebook-only PR.
RUN_MLFLOW is False. Set True only when intentionally logging.
